# GMGI v2 Colab-first agentic setup
Run these cells first in Colab. They mount Drive, install dependencies, start Ollama, configure Diffusers for visual artifact generation, and persist the agentic self-improvement state. This notebook intentionally does not run image-model training or fine-tuning.


In [1]:
# Colab Drive mount and persistent paths
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/gmgi')
except Exception:
    DRIVE_ROOT = Path('experiments/colab_local')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['GMGI_EXPERIENCE_STORE'] = str(DRIVE_ROOT / 'experience_store.jsonl')
os.environ['GMGI_BANDIT_STATE'] = str(DRIVE_ROOT / 'bandit_state.json')
os.environ['GMGI_PROMPT_VERSIONS'] = str(DRIVE_ROOT / 'prompt_versions')
os.environ['GMGI_CREATIVE_BACKEND'] = 'diffusers'
os.environ.setdefault('GMGI_SDXL_MODEL', 'stabilityai/sdxl-turbo')
os.environ.setdefault('GMGI_DISABLE_CLIP_CRITIC', '0')
print('GMGI Drive root:', DRIVE_ROOT)


Mounted at /content/drive
GMGI Drive root: /content/drive/MyDrive/gmgi


In [2]:
# Clone/find/update the repository first, then put it on Python's import path.
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path(os.getenv("PROJECT_DIR", "/content/learning"))
REPO_URL = os.getenv("GMGI_REPO_URL", "https://github.com/kushal-bhargav/learning.git")

REQUIRED_FILES = [
    "src/api/service.py",
    "src/api/app.py",
    "src/evaluation/agent_quality.py",
]

def is_repo_root(path: Path) -> bool:
    return (path / "src" / "api" / "service.py").exists() and (path / "pyproject.toml").exists()

def missing_required(path: Path) -> list[str]:
    return [name for name in REQUIRED_FILES if not (path / name).exists()]

if PROJECT_DIR.exists() and not is_repo_root(PROJECT_DIR):
    candidates = [p for p in [Path("/content/learning"), Path("/content/gift_creator"), Path.cwd()] if is_repo_root(p)]
    candidates += [p for p in Path("/content").glob("*/") if is_repo_root(p)]
    if candidates:
        PROJECT_DIR = candidates[0]
    else:
        raise RuntimeError(f"PROJECT_DIR exists but is not the GMGI repo root: {PROJECT_DIR}. Set PROJECT_DIR to the folder containing src/api/service.py, or delete the folder and rerun clone.")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / ".git").exists():
    print("Updating existing repo checkout...")
    pull = subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], text=True, capture_output=True)
    print(pull.stdout[-2000:])
    if pull.returncode != 0:
        print(pull.stderr[-2000:])
        print("git pull did not complete. Continuing with existing files; required-file check below will catch stale code.")

if not is_repo_root(PROJECT_DIR):
    raise RuntimeError(f"Could not find src/api/service.py under PROJECT_DIR={PROJECT_DIR}")

missing = missing_required(PROJECT_DIR)
if missing:
    raise RuntimeError(
        "This Colab checkout is missing files required by the current notebook: "
        + ", ".join(missing)
        + ". Upload the latest codebase or push/pull the latest repository before running this notebook."
    )

CODE_MARKERS = {
    "src/agents/recommendation.py": "_run_ranked_repair",
    "src/agents/recipient_profiling.py": "_run_profile_repair",
    "src/agents/multi_agent_planning.py": "GMGI_ALLOW_PLANNING_REPAIR",
    "src/agents/gift_intent_reasoning.py": "GMGI_ALLOW_INTENT_REPAIR",
}
missing_markers = []
for rel_path, marker in CODE_MARKERS.items():
    text = (PROJECT_DIR / rel_path).read_text(encoding="utf-8")
    if marker not in text:
        missing_markers.append(f"{rel_path}:{marker}")
if missing_markers:
    raise RuntimeError(
        "This Colab checkout is stale and missing required repair code: "
        + ", ".join(missing_markers)
        + ". Push/upload the latest code, rerun this repo setup cell, then restart backend/frontend."
    )
print("Repair code markers present:", CODE_MARKERS)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
os.environ["PYTHONPATH"] = str(PROJECT_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

print("Project dir:", PROJECT_DIR)
print("Working directory:", Path.cwd())
print("Top-level files:", sorted(p.name for p in PROJECT_DIR.iterdir())[:12])


Project dir: /content/learning
Working directory: /content/learning
Top-level files: ['.git', '.gitignore', 'README.md', 'data', 'eval', 'experiments', 'frontend', 'pyproject.toml', 'run.ipynb', 'scripts', 'specs', 'specs.zip']


In [3]:
# Python dependencies for Ollama-backed agents, Diffusers generation, and the local package.
%pip install -q --upgrade pip
%pip install -q -e ".[dev]" datasets instructor openai "smolagents[litellm]" diffusers transformers accelerate safetensors ollama open-clip-torch pycloudflared pyngrok tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gmgi (pyproject.toml) ... done


In [4]:
# Install, start, and verify Ollama for real local agents.
import os
import shutil
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen

os.environ["GMGI_OLLAMA_MODEL"] = os.getenv("GMGI_OLLAMA_MODEL", "qwen2.5:1.5b")
os.environ["GMGI_OLLAMA_HOST"] = os.getenv("GMGI_OLLAMA_HOST", "http://127.0.0.1:11434")
os.environ["GMGI_OLLAMA_BASE_URL"] = os.getenv("GMGI_OLLAMA_BASE_URL", "http://127.0.0.1:11434/v1")
os.environ["OLLAMA_HOST"] = os.environ["GMGI_OLLAMA_HOST"]
os.environ["GMGI_FORCE_OLLAMA_AGENTS"] = "1"
os.environ["GMGI_ALLOW_AGENT_FALLBACK"] = "0"
os.environ["GMGI_USE_DEMO_AGENT_RESPONSES"] = "0"
os.environ["GMGI_INTENT_METHOD"] = "classifier_hybrid"
os.environ["GMGI_PLANNING_METHOD"] = "rule_constrained"
os.environ["GMGI_ALLOW_INTENT_REPAIR"] = "1"
os.environ["GMGI_ALLOW_PLANNING_REPAIR"] = "1"
os.environ["GMGI_ALLOW_RECIPIENT_REPAIR"] = "1"
os.environ["GMGI_ALLOW_RECOMMENDATION_REPAIR"] = "1"

def run_logged(command: str, *, check: bool = False) -> subprocess.CompletedProcess:
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout, stderr=result.stderr)
    return result

if shutil.which("ollama"):
    print("Ollama binary already available:", shutil.which("ollama"))
else:
    if not shutil.which("zstd"):
        print("Installing zstd, required by the Ollama installer...")
        zstd_install = run_logged("apt-get update -qq && apt-get install -y -qq zstd")
        if zstd_install.returncode != 0 and not shutil.which("zstd"):
            raise RuntimeError("Failed to install zstd, which Ollama requires for extraction. See apt output above.")
    installer = run_logged("curl -fsSL https://ollama.com/install.sh -o /tmp/ollama-install.sh && sh /tmp/ollama-install.sh")
    if installer.returncode != 0:
        print("Ollama installer returned non-zero. Checking whether the binary was still installed...")
    if not shutil.which("ollama"):
        # Some notebook images do not refresh PATH after installing to /usr/local/bin.
        possible = Path("/usr/local/bin/ollama")
        if possible.exists():
            os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
    if not shutil.which("ollama"):
        raise RuntimeError(
            "Ollama install failed and no ollama binary was found. "
            "Review the installer output above; in Colab this is usually a transient network/runtime issue. "
            "Restart runtime and rerun this cell, or switch to a cloud LLM provider in GMGI_LLM_PROVIDER."
        )

print("Ollama binary:", shutil.which("ollama"))
existing = subprocess.run(["pgrep", "-f", "ollama serve"], capture_output=True, text=True)
if existing.returncode != 0:
    log = open("/tmp/ollama.log", "a", encoding="utf-8")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT)
else:
    print("Ollama serve already running with PID(s):", existing.stdout.strip())

def wait_for_ollama(timeout_seconds: int = 120) -> None:
    deadline = time.time() + timeout_seconds
    last_error = None
    url = os.environ["GMGI_OLLAMA_HOST"].rstrip("/") + "/api/tags"
    while time.time() < deadline:
        try:
            with urlopen(url, timeout=5) as response:
                if response.status == 200:
                    return
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    print("Last Ollama log lines:")
    run_logged("tail -120 /tmp/ollama.log || true")
    raise RuntimeError(f"Ollama did not become reachable at {url}: {last_error}")

wait_for_ollama()
pull = run_logged(f"ollama pull {os.environ['GMGI_OLLAMA_MODEL']}")
if pull.returncode != 0:
    raise RuntimeError(f"Failed to pull Ollama model {os.environ['GMGI_OLLAMA_MODEL']}. See output above.")
run_logged("ollama list", check=True)
print("Ollama ready:", os.environ["GMGI_OLLAMA_HOST"], "model:", os.environ["GMGI_OLLAMA_MODEL"])
print("Real-agent mode: demo responses disabled, fallback disabled.")


Installing zstd, required by the Ollama installer...
$ apt-get update -qq && apt-get install -y -qq zstd
Selecting previously unselected package zstd.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...

W: Skipping acquire of configured file 'mai

In [5]:
# Optional: instantiate the service with persistent state paths.
# Ensure project root is active before importing src.* modules.
import os
import sys
from pathlib import Path

PROJECT_DIR = Path(os.getenv("PROJECT_DIR", "/content/learning"))
if not (PROJECT_DIR / "src" / "api" / "service.py").exists():
    raise RuntimeError(f"PROJECT_DIR is not the repo root: {PROJECT_DIR}. Run the repo setup cell first.")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.api.service import AgencyConsoleService

service = AgencyConsoleService(
    experience_store_path=os.environ["GMGI_EXPERIENCE_STORE"],
    bandit_state_path=os.environ["GMGI_BANDIT_STATE"],
    prompt_versions_dir=os.environ["GMGI_PROMPT_VERSIONS"],
)
print("Experience episodes loaded:", len(service.experience_store.episodes))


Experience episodes loaded: 1


In [6]:
# Manual self-improvement trigger after enough completed sessions
from pathlib import Path
from src.agents.prompt_optimizer import PromptOptimizerAgent
from src.agents.llm import create_llm

optimizer = PromptOptimizerAgent(
    llm=create_llm(),
    store=service.experience_store,
    config_dir=Path('src/agents/configs'),
)
print('Updated prompts:', optimizer.run())


Updated prompts: {}


## Agentic evaluation
Run these cells to evaluate agent behavior. The first cell evaluates intent/planning components quickly. The second cell runs real per-agent quality evals through Ollama-backed agents; creative image generation is opt-in with `RUN_CREATIVE_EVAL=1`.


In [7]:
from pathlib import Path
import json
import os
from src.evaluation.intent_planning import ExperimentCase, MethodVariant, compare_intent_planning_methods

config = json.loads(Path("eval/configs/intent_planning_eval.json").read_text(encoding="utf-8"))
cases = [ExperimentCase(**case) for case in config["cases"]]

# Fast/default eval: no LLM calls, no image generation, no training.
# Set RUN_LLM_EVAL=1 only when you intentionally want Ollama-backed variants too.
if os.getenv("RUN_LLM_EVAL", "0") == "1":
    intent_variants = [MethodVariant(**variant) for variant in config["intent_variants"]]
    planning_variants = [MethodVariant(**variant) for variant in config["planning_variants"]]
    run_overall = bool(config.get("run_overall", True))
else:
    intent_variants = [MethodVariant(name="heuristic", method="heuristic")]
    planning_variants = [MethodVariant(name="rule_constrained", method="rule_constrained")]
    run_overall = False

summary = compare_intent_planning_methods(
    cases,
    intent_variants=intent_variants,
    planning_variants=planning_variants,
    output_dir=Path(config.get("output_dir", "experiments/agentic_eval")),
    run_overall=run_overall,
)
print("Cases:", len(cases))
print("Intent variants:", [variant.name for variant in intent_variants])
print("Planning variants:", [variant.name for variant in planning_variants])
print("Component rows:", len(summary["component_rows"]))
print("Overall rows:", len(summary["overall_rows"]))
summary["component_summary"]


Cases: 2
Intent variants: ['heuristic']
Planning variants: ['rule_constrained']
Component rows: 4
Overall rows: 0


{'intent': {'latency_seconds': 0.0016534689999616603,
  'structured_output_valid': 1.0,
  'occasion_exact_or_contains': 1.0,
  'constraint_precision': 0.55,
  'constraint_recall': 1.0,
  'constraint_f1': 0.7083333333333333,
  'clarification_count': 0.5,
  'n': 2},
 'planning': {'latency_seconds': 0.0008991489999630176,
  'structured_output_valid': 1.0,
  'plan_completeness': 1.0,
  'step_ordering_correct': 1.0,
  'dependency_satisfaction': 1.0,
  'executable_plan_rate': 1.0,
  'fallback_present': 1.0,
  'subtask_count': 8.5,
  'n': 2}}

## Per-agent real quality evals
This checks every agent stage with task-specific metrics. It uses real Ollama-backed agents and no demo/fake responses. By default creative image generation is marked skipped to avoid a large Diffusers download; set `RUN_CREATIVE_EVAL=1` to include it.


In [8]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path(os.getenv("PROJECT_DIR", "/content/learning"))
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

script_path = PROJECT_DIR / "src" / "evaluation" / "agent_quality.py"
if not script_path.exists():
    raise RuntimeError(
        f"Missing {script_path}. Your Colab repo checkout is stale. "
        "Rerun the repo setup/update cell, or upload/pull the latest codebase that includes src/evaluation/agent_quality.py."
    )

# Keep this strict: no synthetic/demo backend responses and no fallback masking.
os.environ["GMGI_USE_DEMO_AGENT_RESPONSES"] = "0"
os.environ["GMGI_ALLOW_AGENT_FALLBACK"] = "0"
os.environ["GMGI_FORCE_OLLAMA_AGENTS"] = "1"
os.environ["GMGI_INTENT_METHOD"] = "classifier_hybrid"
os.environ["GMGI_PLANNING_METHOD"] = "rule_constrained"
os.environ["GMGI_ALLOW_INTENT_REPAIR"] = "1"
os.environ["GMGI_ALLOW_PLANNING_REPAIR"] = "1"
os.environ["GMGI_ALLOW_RECIPIENT_REPAIR"] = "1"
os.environ["GMGI_ALLOW_RECOMMENDATION_REPAIR"] = "1"
os.environ["GMGI_CREATIVE_BACKEND"] = "diffusers"
os.environ["GMGI_OLLAMA_TIMEOUT_SECONDS"] = os.getenv("GMGI_OLLAMA_TIMEOUT_SECONDS", "180")
os.environ["GMGI_OLLAMA_NUM_CTX"] = os.getenv("GMGI_OLLAMA_NUM_CTX", "4096")

command = [
    sys.executable,
    "-m",
    "src.evaluation.agent_quality",
    "--config",
    "eval/configs/intent_planning_eval.json",
    "--output-dir",
    "experiments/agent_quality_eval",
    "--limit",
    "2",
]
if os.getenv("RUN_CREATIVE_EVAL", "0") == "1":
    command.append("--include-creative")

print("Running:", " ".join(command))
print("Ollama timeout seconds:", os.environ["GMGI_OLLAMA_TIMEOUT_SECONDS"])
result = subprocess.run(command, text=True, capture_output=True, timeout=900)
print(result.stdout)
if result.stderr:
    print(result.stderr)

import json
report_path = Path("experiments/agent_quality_eval/agent_quality.json")
if not report_path.exists():
    raise RuntimeError("Agent quality eval did not write a report. See command output above.")
report = json.loads(report_path.read_text(encoding="utf-8"))
if result.returncode != 0:
    print("Agent quality eval command exited non-zero, but report was written with available error rows.")
report["summary"]


Running: /usr/bin/python3 -m src.evaluation.agent_quality --config eval/configs/intent_planning_eval.json --output-dir experiments/agent_quality_eval --limit 2
Ollama timeout seconds: 180
{
  "case_pipeline": {
    "n": 2,
    "ok": 0,
    "skipped": 0,
    "mean_quality_score": 0.0
  }
}
Wrote experiments/agent_quality_eval/agent_quality.json

Max retries exceeded. Total attempts: 4, Last error: 2 validation errors for RecipientProfileResponse
output.communication_style
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
output.gift_history_summary
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type



{'case_pipeline': {'n': 2, 'ok': 0, 'skipped': 0, 'mean_quality_score': 0.0}}

## Reference-free logged-session quality evals
This evaluates real sessions already written to `experiments/experience_store.jsonl`. It scores every agent output, cross-component consistency, and behavior metrics from the Agency Ledger actions.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path(os.getenv("PROJECT_DIR", "/content/learning"))
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

store_path = Path(os.getenv("GMGI_EXPERIENCE_STORE", "experiments/experience_store.jsonl"))
if not store_path.exists():
    print(f"No experience store found yet at {store_path}. Run one or more UI sessions first, then rerun this cell.")
else:
    command = [
        sys.executable,
        "-m",
        "src.evals.run",
        "--phase",
        "quality",
        "--store",
        str(store_path),
        "--limit",
        os.getenv("GMGI_LOGGED_EVAL_LIMIT", "20"),
        "--output",
        "experiments/evals/logged_quality.json",
    ]
    print("Running:", " ".join(command))
    result = subprocess.run(command, text=True, capture_output=True, timeout=300)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    report = json.loads(Path("experiments/evals/logged_quality.json").read_text(encoding="utf-8"))
    print(json.dumps(report["summary"], indent=2))


## Generate UI-input permutation dataset
This creates a benchmark set from the exact setup-screen fields: relationship, closeness, occasion, date, budget, formality, preferences, memories, and agency slider. The full default factorial space is large, so the cell writes a deterministic balanced sample.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path(os.getenv("PROJECT_DIR", "/content/learning"))
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

max_cases = os.getenv("GMGI_UI_PERMUTATION_MAX_CASES", "48")
case_file = "experiments/evals/ui_permutation_cases.json"
command = [
    sys.executable,
    "-m",
    "src.evals.run",
    "--phase",
    "ui-permutations",
    "--max-cases",
    max_cases,
    "--output",
    case_file,
]
print("Running:", " ".join(command))
result = subprocess.run(command, text=True, capture_output=True, timeout=120)
print(result.stdout)
if result.stderr:
    print(result.stderr)

payload = json.loads(Path(case_file).read_text(encoding="utf-8"))
print("Generated cases:", len(payload["cases"]))
print("Full factorial size:", payload["metadata"]["full_factorial_size"])
payload["cases"][:3]


## Run real-agent benchmark on UI permutations
This invokes the actual agents against the generated UI-permutation cases. It does not use demo responses. Slow or unavailable LLM/image stages are recorded as timeout/error rows and scored low, which is useful for diagnosing real system readiness. Increase `GMGI_UI_PERMUTATION_EVAL_LIMIT` after a small run succeeds.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path(os.getenv("PROJECT_DIR", "/content/learning"))
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Strict real-system settings: no demo outputs and no fallback masking.
os.environ["GMGI_USE_DEMO_AGENT_RESPONSES"] = "0"
os.environ["GMGI_ALLOW_AGENT_FALLBACK"] = "0"
os.environ["GMGI_FORCE_OLLAMA_AGENTS"] = "1"
os.environ["GMGI_INTENT_METHOD"] = os.getenv("GMGI_INTENT_METHOD", "classifier_hybrid")
os.environ["GMGI_PLANNING_METHOD"] = os.getenv("GMGI_PLANNING_METHOD", "rule_constrained")
os.environ["GMGI_CREATIVE_BACKEND"] = os.getenv("GMGI_CREATIVE_BACKEND", "diffusers")
os.environ["GMGI_OLLAMA_TIMEOUT_SECONDS"] = os.getenv("GMGI_OLLAMA_TIMEOUT_SECONDS", "120")
os.environ["GMGI_OLLAMA_NUM_CTX"] = os.getenv("GMGI_OLLAMA_NUM_CTX", "4096")

case_file = "experiments/evals/ui_permutation_cases.json"
if not Path(case_file).exists():
    raise RuntimeError("Run the UI permutation dataset cell first.")

command = [
    sys.executable,
    "-m",
    "src.evals.run",
    "--phase",
    "benchmark",
    "--case-file",
    case_file,
    "--limit",
    os.getenv("GMGI_UI_PERMUTATION_EVAL_LIMIT", "6"),
    "--stage-timeout",
    os.getenv("GMGI_EVAL_STAGE_TIMEOUT_SECONDS", "45"),
    "--output-dir",
    "experiments/evals/ui_permutation_benchmark",
]
if os.getenv("RUN_CREATIVE_EVAL", "0") == "1":
    command.append("--include-creative")

print("Running:", " ".join(command))
result = subprocess.run(command, text=True, capture_output=True, timeout=int(os.getenv("GMGI_UI_PERMUTATION_TIMEOUT", "1800")))
if result.stdout:
    print(result.stdout[-5000:])
if result.stderr:
    print(result.stderr[-5000:])

report_path = Path("experiments/evals/ui_permutation_benchmark/benchmark_report.json")
if not report_path.exists():
    raise RuntimeError("Benchmark did not write a report. See command output above.")
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report["summary"], indent=2))


## Optional: tabular eval summary
This reads the benchmark CSV into pandas so you can sort weak agents/components quickly inside Colab.


In [ ]:
from pathlib import Path
import pandas as pd

rows_path = Path("experiments/evals/ui_permutation_benchmark/benchmark_rows.csv")
if not rows_path.exists():
    print("No benchmark rows found yet. Run the permutation benchmark cell first.")
else:
    df = pd.read_csv(rows_path)
    display(df[["case_id", "stage", "quality_score", "status"]].sort_values(["quality_score", "stage"]).head(30))
    display(df.groupby("stage", dropna=False)["quality_score"].mean().sort_values())


## Runtime check

GPU is optional for this agentic runner. The backend and agents can run on CPU; Diffusers image generation is faster with GPU.


In [9]:
!python --version

!nvidia-smi || true

!node --version

!npm --version

Python 3.12.13
/bin/bash: line 1: nvidia-smi: command not found
v20.19.0
10.8.2


In [10]:
!npm --prefix frontend install

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 25 packages, and audited 26 packages in 5s
⠴
⠴9 packages are looking for funding
⠴  run `npm fund` for details
⠴
2 high severity vulnerabilities

To address all issues, run:
  npm audit fix

Run `npm audit` for details.
⠴npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠴

## 7. Quick backend/API smoke test


This verifies imports and endpoint behavior before starting long-running servers.

In [11]:
# Real import/config smoke test. This does not use demo agent responses or fake agents.
!GMGI_USE_DEMO_AGENT_RESPONSES=0 GMGI_ALLOW_AGENT_FALLBACK=0 GMGI_FORCE_OLLAMA_AGENTS=1 python -m py_compile src/api/app.py src/api/service.py src/agents/recipient_profiling.py src/agents/relationship_analysis.py src/agents/recommendation.py src/agents/greeting_story.py


## 8. Start the FastAPI backend


The backend listens on `0.0.0.0:8000`. `GMGI_CORS_ORIGIN_REGEX=.*` is set for notebook proxy URLs; keep this open only for notebook demos, not production.

In [12]:
import os
import subprocess
import sys
import time
from pathlib import Path

PROJECT_DIR = Path(os.getenv('PROJECT_DIR', '/content/learning'))
if not (PROJECT_DIR / 'src' / 'api' / 'app.py').exists():
    raise RuntimeError('Run the repo setup cell first; src/api/app.py was not found.')
os.chdir(PROJECT_DIR)

PROCESSES = globals().setdefault('GMGI_PROCESSES', {})

def stop_process(name):
    proc = PROCESSES.get(name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()

stop_process('backend')
backend_env = os.environ.copy()
backend_env['GMGI_CORS_ORIGIN_REGEX'] = '.*'
backend_env['GMGI_CREATIVE_BACKEND'] = 'diffusers'
backend_env['GMGI_FORCE_OLLAMA_AGENTS'] = os.environ.get('GMGI_FORCE_OLLAMA_AGENTS', '1')
backend_env['GMGI_ALLOW_AGENT_FALLBACK'] = '0'
backend_env['GMGI_USE_DEMO_AGENT_RESPONSES'] = '0'
backend_env['GMGI_INTENT_METHOD'] = os.environ.get('GMGI_INTENT_METHOD', 'classifier_hybrid')
backend_env['GMGI_PLANNING_METHOD'] = os.environ.get('GMGI_PLANNING_METHOD', 'rule_constrained')
backend_env['GMGI_ALLOW_INTENT_REPAIR'] = os.environ.get('GMGI_ALLOW_INTENT_REPAIR', '1')
backend_env['GMGI_ALLOW_PLANNING_REPAIR'] = os.environ.get('GMGI_ALLOW_PLANNING_REPAIR', '1')
backend_env['GMGI_ALLOW_RECIPIENT_REPAIR'] = os.environ.get('GMGI_ALLOW_RECIPIENT_REPAIR', '1')
backend_env['GMGI_ALLOW_RECOMMENDATION_REPAIR'] = os.environ.get('GMGI_ALLOW_RECOMMENDATION_REPAIR', '1')
backend_env['GMGI_EXPERIENCE_STORE'] = os.environ['GMGI_EXPERIENCE_STORE']
backend_env['GMGI_BANDIT_STATE'] = os.environ['GMGI_BANDIT_STATE']
backend_env['GMGI_PROMPT_VERSIONS'] = os.environ['GMGI_PROMPT_VERSIONS']
backend_env.setdefault('GMGI_OLLAMA_MODEL', os.environ.get('GMGI_OLLAMA_MODEL', 'qwen2.5:1.5b'))
backend_env.setdefault('GMGI_OLLAMA_HOST', os.environ.get('GMGI_OLLAMA_HOST', 'http://127.0.0.1:11434'))
backend_env.setdefault('GMGI_OLLAMA_BASE_URL', os.environ.get('GMGI_OLLAMA_BASE_URL', 'http://127.0.0.1:11434/v1'))
backend_env.setdefault('OLLAMA_MODEL', backend_env.get('GMGI_OLLAMA_MODEL', 'qwen2.5:1.5b'))
backend_env.setdefault('OLLAMA_HOST', backend_env.get('GMGI_OLLAMA_HOST', 'http://127.0.0.1:11434'))


# Refuse to start the backend if Ollama is not reachable; otherwise the UI would show connection-refused errors.
from urllib.request import urlopen
ollama_tags_url = backend_env['GMGI_OLLAMA_HOST'].rstrip('/') + '/api/tags'
try:
    with urlopen(ollama_tags_url, timeout=10) as response:
        if response.status != 200:
            raise RuntimeError(f'Ollama health returned status {response.status}')
except Exception as exc:
    raise RuntimeError(f'Ollama is not reachable at {ollama_tags_url}. Re-run the Ollama setup cell before starting backend.') from exc

backend = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'src.api.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=backend_env,
)
PROCESSES['backend'] = backend
time.sleep(5)
print('Working directory:', Path.cwd())
print('Backend PID:', backend.pid, 'alive:', backend.poll() is None)
print('Creative backend:', backend_env['GMGI_CREATIVE_BACKEND'])
print('Demo responses:', backend_env['GMGI_USE_DEMO_AGENT_RESPONSES'], 'Fallback:', backend_env['GMGI_ALLOW_AGENT_FALLBACK'])
print('Intent method:', backend_env['GMGI_INTENT_METHOD'], 'Planning method:', backend_env['GMGI_PLANNING_METHOD'])
print('Intent repair:', backend_env['GMGI_ALLOW_INTENT_REPAIR'], 'Planning repair:', backend_env['GMGI_ALLOW_PLANNING_REPAIR'])
print('Recipient repair:', backend_env['GMGI_ALLOW_RECIPIENT_REPAIR'], 'Recommendation repair:', backend_env['GMGI_ALLOW_RECOMMENDATION_REPAIR'])


Working directory: /content/learning
Backend PID: 7037 alive: True
Creative backend: diffusers
Training/checkpoint requirement: 0
Demo responses: 0 Fallback: 0
Intent method: classifier_hybrid Planning method: rule_constrained
Intent repair: 1 Planning repair: 1


In [13]:
import json
import time
from urllib.request import urlopen

def get_json(path: str, timeout_seconds: int = 60):
    deadline = time.time() + timeout_seconds
    last_error = None
    url = f"http://127.0.0.1:8000{path}"
    while time.time() < deadline:
        try:
            with urlopen(url, timeout=10) as response:
                raw = response.read().decode("utf-8")
            return json.loads(raw)
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{path} did not return JSON: {last_error}")

health = get_json("/health")
print(json.dumps(health, indent=2))
if not health.get("ollama_ready"):
    raise RuntimeError("Backend is running but Ollama is not reachable. Re-run the Ollama setup cell, then restart backend.")
if health.get("demo_responses") or health.get("agent_fallback"):
    raise RuntimeError("Backend is not in strict real-agent mode. Demo responses and fallback must both be false.")
print(json.dumps(get_json("/personas"), indent=2)[:4000])


{
  "status": "ok",
  "ollama_ready": true,
  "ollama_host": "http://127.0.0.1:11434",
  "ollama_error": null,
  "demo_responses": false,
  "agent_fallback": false,
  "creative_backend": "diffusers"
}
[
  {
    "persona_id": "custom-live",
    "label": "Create a live gifting context",
    "synthetic": false,
    "occasions": []
  }
]


## 9. Prepare Public URL Mode

The frontend must start before Cloudflare/ngrok creates the public URL. This cell only sets tunnel mode and same-origin API proxy settings.

In [14]:
import os

PUBLIC_TUNNEL = os.getenv("PUBLIC_TUNNEL", "cloudflare").lower()
BACKEND_URL = ""  # Use Vite same-origin proxy: /personas, /sessions, /artifacts, /health.
FRONTEND_URL = None  # Created later, after Vite is running.

print("Public tunnel mode:", PUBLIC_TUNNEL)
print("Frontend API base:", BACKEND_URL or "same-origin Vite proxy")
print("Run the frontend start cell next, then run the public URL cell after it.")

Public tunnel mode: cloudflare
Frontend API base: same-origin Vite proxy
Run the frontend start cell next, then run the public URL cell after it.


## 10. Start the React/Vite Agency Console


The frontend is started with `VITE_API_BASE` pointing at the backend URL above, so browser requests reach the notebook VM instead of your local laptop.

In [15]:
stop_process("frontend")
frontend_env = os.environ.copy()
frontend_env["VITE_API_BASE"] = BACKEND_URL
frontend = subprocess.Popen(
    ["npm", "--prefix", "frontend", "run", "dev", "--", "--host", "0.0.0.0", "--port", "5173"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=frontend_env,
)
PROCESSES["frontend"] = frontend
time.sleep(8)
print("Frontend PID:", frontend.pid, "alive:", frontend.poll() is None)
print("Frontend API base:", BACKEND_URL or "same-origin Vite proxy")
print("Public URL is not created in this cell. Run the next public URL cell after this one.")

Frontend PID: 7069 alive: True
Frontend API base: same-origin Vite proxy
Public URL is not created in this cell. Run the next public URL cell after this one.


## 11. Create And Open Public Frontend URL

Run this only after the frontend PID cell says `alive: True`. The cell waits until the public URL responds before printing it.

In [16]:
import re
import subprocess
import time
from urllib.request import Request, urlopen

def wait_for_public_url(url: str, timeout_seconds: int = 120) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            req = Request(url, headers={"User-Agent": "GMGI-Colab-Healthcheck"})
            with urlopen(req, timeout=10) as response:
                if response.status < 500:
                    return True
        except Exception:
            time.sleep(3)
    return False

def drain_process_log(name: str, lines: int = 80) -> None:
    proc = PROCESSES.get(name)
    if not proc or proc.stdout is None:
        print(f"No process/log for {name}")
        return
    collected = []
    while True:
        line = proc.stdout.readline()
        if not line:
            break
        collected.append(line.rstrip())
        if len(collected) >= lines:
            break
    print("\n".join(collected[-lines:]) or f"No new {name} log lines.")

if "frontend" not in globals() or frontend.poll() is not None:
    drain_process_log("frontend")
    raise RuntimeError("Frontend process is not running. Re-run the frontend start cell first.")

if PUBLIC_TUNNEL == "ngrok":
    from pyngrok import ngrok
    token = os.getenv("NGROK_AUTHTOKEN", "").strip()
    if not token:
        raise RuntimeError("Set NGROK_AUTHTOKEN or switch PUBLIC_TUNNEL to cloudflare.")
    ngrok.set_auth_token(token)
    ngrok.kill()
    FRONTEND_URL = ngrok.connect(5173, "http").public_url.rstrip("/")
else:
    subprocess.run(["wget", "-q", "-O", "/tmp/cloudflared.deb", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
    subprocess.run(["dpkg", "-i", "/tmp/cloudflared.deb"], check=True)
    old = PROCESSES.get("cloudflared")
    if old and old.poll() is None:
        old.terminate()
    cloudflared = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5173", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    PROCESSES["cloudflared"] = cloudflared
    FRONTEND_URL = None
    log_lines = []
    for _ in range(90):
        line = cloudflared.stdout.readline() if cloudflared.stdout else ""
        if line:
            log_lines.append(line.rstrip())
        match = re.search(r"https://[-a-zA-Z0-9.]+trycloudflare.com", line)
        if match:
            FRONTEND_URL = match.group(0).rstrip("/")
            break
        time.sleep(1)
    if not FRONTEND_URL:
        print("Cloudflared logs:")
        print("\n".join(log_lines[-80:]))
        raise RuntimeError("Cloudflare tunnel did not print a public URL.")

print("Waiting for public DNS/HTTP readiness...")
if not wait_for_public_url(FRONTEND_URL):
    print("URL was created but did not become reachable yet. Wait 30 seconds and run this cell again.")
else:
    print("Public URL is reachable.")
print("Open this public URL in a new tab:", FRONTEND_URL)
print("The backend is not separately public; Vite proxies API routes from the same public frontend URL.")

Waiting for public DNS/HTTP readiness...
Public URL is reachable.
Open this public URL in a new tab: https://humanitarian-continent-moms-moving.trycloudflare.com
The backend is not separately public; Vite proxies API routes from the same public frontend URL.


## 12. End-to-end API request example


This creates one backend session from the notebook, useful when debugging before touching the UI.

In [17]:
import json
from urllib.request import Request, urlopen

payload = {
    "persona_id": "custom-live",
    "agency_slider": 0.5,
    "seed": 2026,
    "custom_profile": {
        "giver_name": "Asha",
        "recipient_name": "Mira",
        "relationship_type": "friend",
        "closeness_score": 4,
        "occasion_name": "Birthday",
        "occasion_date": "2026-12-18",
        "budget_hint": "USD 60-100",
        "formality": "casual",
        "preferences": ["tea", "ceramics", "green"],
        "memories": ["We found a tiny tea shop after getting lost."],
    },
}
request = Request(
    "http://127.0.0.1:8000/sessions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urlopen(request, timeout=30) as response:
    created = json.loads(response.read().decode("utf-8"))
print(json.dumps(created, indent=2)[:4000])


{
  "session_id": "live-4d883759-2cddcbfe",
  "giver_id": "person-giver-4d883759",
  "recipient_id": "person-recipient-4d883759",
  "occasion_id": "occasion-live-4d883759",
  "stage_log": [],
  "next_stage": "recipient_profiling",
  "ledger": {
    "session_id": "live-4d883759-2cddcbfe",
    "timeline": [],
    "counts": {
      "accept": 0,
      "edit": 0,
      "regenerate": 0,
      "delegate": 0,
      "pending": 0,
      "error": 0
    },
    "authorship": "hybrid",
    "stage_count": 8,
    "completed": false
  }
}


## 13. Server logs and shutdown


Use the log cell when something fails. Run shutdown before restarting ports or ending the notebook.

In [ ]:
def show_process_log(name, lines=120):

    proc = PROCESSES.get(name)

    if not proc or proc.stdout is None:

        print(f"No process/log for {name}")

        return

    collected = []

    while True:

        line = proc.stdout.readline()

        if not line:

            break

        collected.append(line.rstrip())

        if len(collected) >= lines:

            break

    print("\n".join(collected[-lines:]) or f"No new {name} log lines.")


show_process_log("backend")

show_process_log("frontend")

In [ ]:
# # Run when finished or before restarting the app.
# stop_process("cloudflared")
# stop_process("frontend")
# stop_process("backend")
# print("Stopped cloudflared/backend/frontend processes.")